# 1. Preparação Geral dos Dados e Constantes

Nesta etapa, carregamos a base limpa do NHIS 2025 (`adult25_limpo.csv`), definimos o `ID_DOMICILIO` como índice e isolamos a variável alvo multiclasse (`SEVERIDADE_DEPRESSAO_PHQ`), filtrando os registros indefinidos na Target (valores `-1` e `5`).

Além disso, isolamos a coluna `PESO_AMOSTRAL` e removemos do conjunto de preditores ($X$) todas as colunas de sintomas diretos de depressão para evitar vazamento de dados (*data leakage*).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ==============================================================================
# CONFIGURAÇÃO DE VARIÁVEIS GLOBAIS / CONSTANTES
# ==============================================================================
ID_COLUNA = 'ID_DOMICILIO'
PESO_COLUNA = 'PESO_AMOSTRAL'
TARGET = 'SEVERIDADE_DEPRESSAO_PHQ'

COLUNAS_SINTOMAS_DEPRESSAO = [
    'FREQ_PERDA_INTERESSE_2SEM',
    'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM',
    'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM',
    'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM',
    'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO',
    'REMEDIO_DEPRESSAO',
    'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO'
]

# 1. Carregar a base limpa e definir o ID como índice
df = pd.read_csv('adult25_limpo.csv').set_index(ID_COLUNA)

# 2. Eliminar os registros indefinidos no TARGET (-1 e 5)
df = df[~df[TARGET].isin([-1, 5])].copy()

# 3. Isolar Target (y), Peso Amostral e Features Preditoras (X)
y = df[TARGET]
sample_weights = df[PESO_COLUNA]
colunas_remover = [TARGET, PESO_COLUNA] + COLUNAS_SINTOMAS_DEPRESSAO
X = df.drop(columns=[col for col in colunas_remover if col in df.columns])

# 2. Cenário A: Treinamento Mantendo Códigos Indeterminados (-1) nas Features

Neste primeiro cenário, mantemos as categorias de não-resposta (`-1`) presentes nas variáveis preditoras ($X$), permitindo que o modelo de árvore e a codificação One-Hot tratem o valor como uma categoria válida.

In [2]:
# Split Estratificado
X_train_A, X_test_A, y_train_A, y_test_A, w_train_A, w_test_A = train_test_split(
    X, y, sample_weights, test_size=0.2, random_state=42, stratify=y
)

# Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA']
cat_cols = [col for col in X_train_A.columns if col not in num_cols]

preprocessor_A = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_train_proc_A = preprocessor_A.fit_transform(X_train_A)
X_test_proc_A = preprocessor_A.transform(X_test_A)

# Modelo Random Forest
model_A = RandomForestClassifier(random_state=42, class_weight='balanced')
model_A.fit(X_train_proc_A, y_train_A, sample_weight=w_train_A)

# Avaliação Cenário A
y_pred_A = model_A.predict(X_test_proc_A)
y_proba_A = model_A.predict_proba(X_test_proc_A)

print("--- CENÁRIO A: Relatório de Classificação (Com -1 nas Features) ---")
print(classification_report(y_test_A, y_pred_A, sample_weight=w_test_A, zero_division=0))

print("--- CENÁRIO A: Matriz de Confusão 4x4 ---")
print(np.round(confusion_matrix(y_test_A, y_pred_A, sample_weight=w_test_A)).astype(int))
print(f"ROC-AUC Score (OvR): {roc_auc_score(y_test_A, y_proba_A, multi_class='ovr', sample_weight=w_test_A):.4f}")

--- CENÁRIO A: Relatório de Classificação (Com -1 nas Features) ---
              precision    recall  f1-score   support

           1       0.94      0.89      0.91 40224045.76199999
           2       0.44      0.61      0.51 7209618.388000002
           3       0.44      0.25      0.32 2466480.354
           4       0.63      0.64      0.63 1334503.7569999998

    accuracy                           0.82 51234648.26099999
   macro avg       0.61      0.60      0.59 51234648.26099999
weighted avg       0.83      0.82      0.82 51234648.26099999

--- CENÁRIO A: Matriz de Confusão 4x4 ---
[[35900034  4173661   128854    21497]
 [ 2268874  4405756   425006   109983]
 [  142188  1321019   627418   375855]
 [   29742   198955   256571   849235]]
ROC-AUC Score (OvR): 0.9140


# 3. Cenário B: Treinamento Apenas com Informações Completas (Remoção de -1 no X)

Conforme discutido com a equipe e orientação do prof. Roy, realizamos um teste estrito filtrando qualquer observação que contenha não-resposta (`-1`) em qualquer feature preditora. A amostragem retida foi de **22.084 registros (93.5% dos dados)**.

In [3]:
# Filtragem Estrita das Features
mascara_sem_menos1 = (X != -1).all(axis=1)
X_strict = X[mascara_sem_menos1].copy()
y_strict = y[mascara_sem_menos1].copy()
w_strict = sample_weights[mascara_sem_menos1].copy()

print(f"Total de registros na base completa estrita: {len(X_strict)}")

# Split Estratificado Cenário B
X_train_B, X_test_B, y_train_B, y_test_B, w_train_B, w_test_B = train_test_split(
    X_strict, y_strict, w_strict, test_size=0.2, random_state=42, stratify=y_strict
)

# Pré-processamento
preprocessor_B = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_train_proc_B = preprocessor_B.fit_transform(X_train_B)
X_test_proc_B = preprocessor_B.transform(X_test_B)

# Modelo Random Forest Cenário B
model_B = RandomForestClassifier(random_state=42, class_weight='balanced')
model_B.fit(X_train_proc_B, y_train_B, sample_weight=w_train_B)

# Avaliação Cenário B
y_pred_B = model_B.predict(X_test_proc_B)
y_proba_B = model_B.predict_proba(X_test_proc_B)

print("--- CENÁRIO B: Relatório de Classificação (Sem -1 nas Features) ---")
print(classification_report(y_test_B, y_pred_B, sample_weight=w_test_B, zero_division=0))

print("--- CENÁRIO B: Matriz de Confusão 4x4 ---")
print(np.round(confusion_matrix(y_test_B, y_pred_B, sample_weight=w_test_B)).astype(int))
print(f"ROC-AUC Score (OvR): {roc_auc_score(y_test_B, y_proba_B, multi_class='ovr', sample_weight=w_test_B):.4f}")

Total de registros na base completa estrita: 22084
--- CENÁRIO B: Relatório de Classificação (Sem -1 nas Features) ---
              precision    recall  f1-score   support

           1       0.94      0.89      0.91 37773204.54699994
           2       0.42      0.60      0.49 6824341.566000004
           3       0.28      0.19      0.23 2187297.4779999997
           4       0.62      0.52      0.57 1212457.585

    accuracy                           0.81 47997301.17599995
   macro avg       0.56      0.55      0.55 47997301.17599995
weighted avg       0.83      0.81      0.81 47997301.17599995

--- CENÁRIO B: Matriz de Confusão 4x4 ---
[[33571356  3965806   230880     5162]
 [ 2156776  4069561   515537    82468]
 [  116638  1355470   420541   294648]
 [       0   233664   349171   629622]]
ROC-AUC Score (OvR): 0.9055


# 4. Otimização de Hiperparâmetros no Random Forest (RandomizedSearchCV)

Para buscar um melhor equilíbrio entre as classes de severidade e reduzir a sobreposição entre as classes Leve (2) e Moderada (3), aplicamos o `RandomizedSearchCV`. 

Buscamos otimizar a profundidade máxima das árvores (`max_depth`), o número de estimadores (`n_estimators`), e o tamanho mínimo de amostras em nós/folhas (`min_samples_split` e `min_samples_leaf`) utilizando validação cruzada estratificada em 3 folds.

In [5]:
from sklearn.model_selection import RandomizedSearchCV


# 1. Definir o Espaço de Busca de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, 25, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', 0.3],
    'class_weight': ['balanced', 'balanced_subsample']
}

# 2. Inicializar o Modelo Base
rf_base = RandomForestClassifier(random_state=42)

# 3. Configurar a Busca Aleatória (3 folds de validação cruzada)
random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=15,                   # Testa 15 combinações aleatórias
    scoring='f1_macro',          # Foca em melhorar a média macro entre as 4 classes
    cv=3,
    random_state=42,
    n_jobs=-1,                   # Usa todos os núcleos do processador
    verbose=1
)

# 4. Executar a Busca no Cenário B (Base Estritamente Limpa)
# Nota: Passamos o sample_weight do treino na busca
random_search.fit(X_train_proc_B, y_train_B, sample_weight=w_train_B)

print("\n--- Melhores Hiperparâmetros Encontrados ---")
print(random_search.best_params_)

# 5. Avaliar o Melhor Modelo no Conjunto de Teste
best_model = random_search.best_estimator_

y_pred_tuned = best_model.predict(X_test_proc_B)
y_proba_tuned = best_model.predict_proba(X_test_proc_B)

print("\n--- RELATÓRIO APÓS TUNING (Cenário B - 4 Classes) ---")
print(classification_report(y_test_B, y_pred_tuned, sample_weight=w_test_B, zero_division=0))

print("--- MATRIZ DE CONFUSÃO APÓS TUNING (4x4) ---")
print(np.round(confusion_matrix(y_test_B, y_pred_tuned, sample_weight=w_test_B)).astype(int))

print(f"\nROC-AUC Score (OvR) Otimizado: {roc_auc_score(y_test_B, y_proba_tuned, multi_class='ovr', sample_weight=w_test_B):.4f}")

Fitting 3 folds for each of 15 candidates, totalling 45 fits

--- Melhores Hiperparâmetros Encontrados ---
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 25, 'class_weight': 'balanced'}

--- RELATÓRIO APÓS TUNING (Cenário B - 4 Classes) ---
              precision    recall  f1-score   support

           1       0.94      0.87      0.91 37773204.54699994
           2       0.41      0.61      0.49 6824341.566000004
           3       0.34      0.28      0.31 2187297.4779999997
           4       0.57      0.57      0.57 1212457.585

    accuracy                           0.80 47997301.17599995
   macro avg       0.57      0.58      0.57 47997301.17599995
weighted avg       0.83      0.80      0.81 47997301.17599995

--- MATRIZ DE CONFUSÃO APÓS TUNING (4x4) ---
[[32944775  4577413   235503    15513]
 [ 1930933  4142939   564842   185628]
 [   96528  1158429   616685   315656]
 [       0   143414   380788   688256]]

ROC-AUC Sco

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Carregar a base limpa
df = pd.read_csv('adult25_limpo.csv').set_index('ID_DOMICILIO')

# 2. Remover indeterminados (-1 e 5) da Target PHQ original
df = df[~df['SEVERIDADE_DEPRESSAO_PHQ'].isin([-1, 5])].copy()

# 3. Aplicar a transformação binária (Sugestão do Alexandre)
# 0 = Sem Depressão (Nível 1) | 1 = Com Depressão (Níveis 2, 3 e 4)
df['DEPRESSAO_PHQ_SIMPLIFICADO'] = (df['SEVERIDADE_DEPRESSAO_PHQ'] != 1).astype(int)

TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
PESO_COLUNA = 'PESO_AMOSTRAL'

COLUNAS_SINTOMAS = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ' # Removemos a target original do X
]

# 4. Isolar X, y e Pesos nas Features Completas (sem -1 nas preditoras)
y_bin = df[TARGET_BIN]
weights = df[PESO_COLUNA]
X_base = df.drop(columns=[col for col in [TARGET_BIN, PESO_COLUNA] + COLUNAS_SINTOMAS if col in df.columns])

# Filtro de registros completos (22.084 linhas)
mascara_completos = (X_base != -1).all(axis=1)
X_bin = X_base[mascara_completos].copy()
y_bin = y_bin[mascara_completos].copy()
w_bin = weights[mascara_completos].copy()

# 5. Split Estratificado Binário
X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
    X_bin, y_bin, w_bin, test_size=0.2, random_state=42, stratify=y_bin
)

# 6. Pré-processamento
num_cols = ['RAZAO_RENDA_POBREZA']
cat_cols = [c for c in X_tr.columns if c not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

X_tr_proc = preprocessor.fit_transform(X_tr)
X_te_proc = preprocessor.transform(X_te)

# 7. Treinar Random Forest Binário (com hiperparâmetros otimizados)
rf_binario = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_split=5,
    max_features='log2',
    class_weight='balanced',
    random_state=42
)
rf_binario.fit(X_tr_proc, y_tr, sample_weight=w_tr)

# 8. Avaliação do Modelo Binário
y_pred = rf_binario.predict(X_te_proc)
y_proba = rf_binario.predict_proba(X_te_proc)[:, 1]

print("--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (RANDOM FOREST) ---")
print(classification_report(y_te, y_pred, sample_weight=w_te, zero_division=0))

print("--- MATRIZ DE CONFUSÃO BINÁRIA (2x2) ---")
print(np.round(confusion_matrix(y_te, y_pred, sample_weight=w_te)).astype(int))

print(f"\nROC-AUC Score (Binário): {roc_auc_score(y_te, y_proba, sample_weight=w_te):.4f}")

--- RELATÓRIO DE CLASSIFICAÇÃO BINÁRIO (RANDOM FOREST) ---
              precision    recall  f1-score   support

           0       0.93      0.89      0.91 37773204.546999976
           1       0.66      0.75      0.70 10425277.420000015

    accuracy                           0.86 48198481.96699999
   macro avg       0.79      0.82      0.80 48198481.96699999
weighted avg       0.87      0.86      0.86 48198481.96699999

--- MATRIZ DE CONFUSÃO BINÁRIA (2x2) ---
[[33671240  4101965]
 [ 2615251  7810026]]

ROC-AUC Score (Binário): 0.9135
